# 1、使用@tool的方式创建工具

In [11]:
from pydantic import BaseModel


#定义一个函数

def add_number(num1: int, num2: int) -> int:
    """计算两个数的和"""
    return num1 + num2


add_number(10, 20)

30

如何定义一个工具


In [12]:
from langchain_core.tools import tool, StructuredTool


@tool
def add_number(num1: int, num2: int) -> int:
    """计算两个数的和"""
    return num1 + num2


print(f"name = {add_number.name}")
print(f"description = {add_number.description}")
print(f"args = {add_number.args}")
print(f"return_direct= {add_number.return_direct}")

name = add_number
description = 计算两个数的和
args = {'num1': {'title': 'Num1', 'type': 'integer'}, 'num2': {'title': 'Num2', 'type': 'integer'}}
return_direct= False


In [13]:
from langchain_core.tools import tool


#使用装饰器修改函数的一些属性信息

@tool(return_direct= True)
def add_number(num1: int, num2: int) -> int:
    """计算两个数的和"""
    return num1 + num2


print(f"name = {add_number.name}")
print(f"description = {add_number.description}")
print(f"args = {add_number.args}")
print(f"return_direct= {add_number.return_direct}")

name = add_number
description = 计算两个数的和
args = {'num1': {'title': 'Num1', 'type': 'integer'}, 'num2': {'title': 'Num2', 'type': 'integer'}}
return_direct= True


In [14]:
from pydantic import Field
from langchain_core.tools import tool
from pydantic.main import BaseModel


class FieldInfo(BaseModel):
    num1: int = Field(description="第1个参数")
    num2: int = Field(description="第2个参数")


#使用装饰器修改函数的一些属性信息
@tool(args_schema=FieldInfo)
def add_number(num1: int, num2: int) -> int:
    """计算两个数的和"""
    return num1 + num2


print(f"name = {add_number.name}")
print(f"description = {add_number.description}")
print(f"args = {add_number.args}")
print(f"return_direct= {add_number.return_direct}")

name = add_number
description = 计算两个数的和
args = {'num1': {'description': '第1个参数', 'title': 'Num1', 'type': 'integer'}, 'num2': {'description': '第2个参数', 'title': 'Num2', 'type': 'integer'}}
return_direct= False


# 2、使用StructuredTool.from_function定义一个工具



In [15]:
from langchain_core.tools.structured import StructuredTool

class FieldInfo(BaseModel):
    num1: int = Field(description="第1个参数")
    num2: int = Field(description="第2个参数")

def add_number(num1: int, num2: int) -> int:
    """计算两个数的和"""
    return num1 + num2


#创建工具的方式2
tool = StructuredTool.from_function(
    func=add_number,
    name="add_two_number",
    description="计算两个整数的和",#改为了计算两个整数的和
    return_direct=True,
    args_schema=FieldInfo,
)

print(f"name = {tool.name}")
print(f"description = {tool.description}")
print(f"args = {tool.args}")
print(f"return_direct= {tool.return_direct}")

name = add_two_number
description = 计算两个整数的和
args = {'num1': {'description': '第1个参数', 'title': 'Num1', 'type': 'integer'}, 'num2': {'description': '第2个参数', 'title': 'Num2', 'type': 'integer'}}
return_direct= True


使用工具的应用举例

In [16]:
# 1.导入相关依赖
# 从langchain社区工具中导入文件移动工具
from langchain_community.tools import MoveFileTool
# 从langchain核心消息模块导入人类消息类
from langchain_core.messages import HumanMessage
# 从langchain核心工具调用工具中导入将工具转换为OpenAI函数的函数
from langchain_core.utils.function_calling import convert_to_openai_function
# 从langchain_openai模块导入ChatOpenAI类
from langchain_openai import ChatOpenAI
# 导入操作系统接口模块
import os
# 导入dotenv模块用于加载环境变量
import dotenv
# 再次导入ChatOpenAI（可能是冗余导入）
from langchain_openai import ChatOpenAI

# 加载.env文件中的环境变量
dotenv.load_dotenv()

# 从环境变量中获取OpenAI API密钥并设置为环境变量
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
# 从环境变量中获取OpenAI基础URL并设置为环境变量
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 2.定义LLM模型
# 创建ChatOpenAI实例，使用gpt-4o-mini模型，温度参数设为0（确定性输出）
model =ChatOpenAI(model="gpt-4o-mini",temperature=0)

# 3.定义工具
# 创建工具列表，包含文件移动工具
tools = [MoveFileTool()]

# 4.将工具转换为函数
# 将工具列表中的每个工具转换为OpenAI函数格式
functions = [convert_to_openai_function(t) for t in tools]

# print(functions[0])

# 4.模型使用函数
# 调用模型，传入人类消息和函数列表
message = model.invoke(
    [HumanMessage(content="move file foo to bar")],
    functions=functions
)

# 打印模型返回的消息
print(message)

content='' additional_kwargs={'function_call': {'arguments': '{"source_path":"foo","destination_path":"bar"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 74, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'finish_reason': 'function_call', 'logprobs': None} id='run--64a6c6b9-d5fa-4599-8215-bfcf4b0c7400-0' usage_metadata={'input_tokens': 74, 'output_tokens': 21, 'total_tokens': 95, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [17]:
message = model.invoke(
    [HumanMessage(content="今天的天气怎么样？")],
    functions=functions
)

print(message) #没有天气工具，所以无法获取天气

content='抱歉，我无法提供实时天气信息。你可以通过天气预报网站或应用程序查看今天的天气情况。需要其他帮助吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 74, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'finish_reason': 'stop', 'logprobs': None} id='run--f3a81a79-c4e1-4b14-86a7-373e87fe9738-0' usage_metadata={'input_tokens': 74, 'output_tokens': 32, 'total_tokens': 106, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


确认是否存在我们要调用的函数

In [18]:
message = model.invoke(
    [HumanMessage(content="将本目录下的abc.txt文件移动到C:\\Users\\41507\\Desktop")],
    functions=functions
)

print(message)

content='' additional_kwargs={'function_call': {'arguments': '{"source_path":"abc.txt","destination_path":"C:\\\\Users\\\\41507\\\\Desktop\\\\abc.txt"}', 'name': 'move_file'}, 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 87, 'total_tokens': 119, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'finish_reason': 'function_call', 'logprobs': None} id='run--2ad970c3-3cee-45db-91d9-125fdde2ca9f-0' usage_metadata={'input_tokens': 87, 'output_tokens': 32, 'total_tokens': 119, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [19]:
import json  # 导入json模块，用于解析JSON字符串

# 检查message的additional_kwargs中是否包含"function_call"键
if "function_call" in message.additional_kwargs:
    # 提取工具名称
    tool_name = message.additional_kwargs["function_call"]["name"]
    # 解析工具调用参数（从JSON字符串转为字典）
    tool_args = json.loads(message.additional_kwargs["function_call"]["arguments"])
    # 打印工具调用信息
    print(f"调用工具: {tool_name}, 参数: {tool_args}")
else:
    # 如果没有工具调用，直接打印模型回复内容
    print("模型回复:", message.content)

调用工具: move_file, 参数: {'source_path': 'abc.txt', 'destination_path': 'C:\\Users\\41507\\Desktop\\abc.txt'}


In [20]:
# 从 langchain.tools 模块导入 MoveFileTool 工具类
from langchain.tools import MoveFileTool

# 判断 message 的附加参数字典中函数调用名称是否包含 "move_file"
if "move_file" in message.additional_kwargs["function_call"]["name"]:
    # 实例化 MoveFileTool 工具
    tool = MoveFileTool()
    # 执行工具并传入参数，返回结果
    result = tool.run(tool_args)  # 执行工具
    # 打印工具执行后的结果
    print("工具执行结果:", result)

工具执行结果: File moved successfully from abc.txt to C:\Users\41507\Desktop\abc.txt.
